# 🎙️ Qwen3-TTS — Complete Google Colab Notebook

> **Qwen3-TTS** is Alibaba's state-of-the-art open-weight TTS series (Jan 2026).  
> Supports **Voice Cloning**, **Voice Design**, and **Custom Preset Voices** across 10 languages — fully free, no API key needed.

---

## 📋 Table of Contents
| Section | Description |
|---|---|
| [0 — Runtime Check](#section0) | Verify GPU & VRAM |
| [1 — Install Dependencies](#section1) | `qwen-tts`, `soundfile`, `gradio` |
| [2 — Configuration](#section2) | Set model size, dtype, flash attention |
| [3 — Custom Voice](#section3) | Preset speakers (Ryan, Vivian, …) |
| [4 — Voice Design](#section4) | Describe a voice in plain English |
| [5 — Voice Cloning](#section5) | Clone any voice from ~3s reference audio |
| [6 — Design → Clone](#section6) | Advanced: design a voice, then clone it |
| [7 — Gradio Web UI](#section7) | Launch an interactive UI with a public link |
| [8 — Save to Drive](#section8) | Download or save all WAVs to Google Drive |

---

### ⚡ Quick Start
1. **Runtime → Change runtime type → T4 GPU** (free tier works!)
2. Run **Section 0** to confirm your GPU
3. Run **Section 1** to install packages
4. Run **Section 2** to configure settings
5. Jump to any section you want to try!

> **💡 Tip:** Each section that loads a model first clears the previous model from VRAM automatically. Just run cells in order within each section.

---
## Section 0 — 🚀 Runtime Check
Run this cell first to confirm you have a GPU. If not, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import subprocess
import sys

print("=" * 60)
print("  Qwen3-TTS — Environment Check")
print("=" * 60)

# Python version
print(f"\n🐍 Python: {sys.version.split()[0]}")

# GPU check
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                         "--format=csv,noheader,nounits"],
                        capture_output=True, text=True)

if result.returncode != 0 or result.stdout.strip() == "":
    print("\n⚠️  NO GPU DETECTED!")
    print("   Please go to: Runtime → Change runtime type → Hardware accelerator → GPU (T4)")
    print("   Then re-run this cell.")
else:
    lines = result.stdout.strip().split("\n")
    print("\n🖥️  GPU(s) Available:")
    for i, line in enumerate(lines):
        parts = [p.strip() for p in line.split(",")]
        name, total, free = parts[0], float(parts[1]), float(parts[2])
        used = total - free
        pct  = used / total * 100
        print(f"   GPU {i}: {name}")
        print(f"          Total VRAM : {total/1024:.1f} GB")
        print(f"          Used  VRAM : {used/1024:.1f} GB  ({pct:.0f}%)")
        print(f"          Free  VRAM : {free/1024:.1f} GB")

    # Check if PyTorch sees CUDA
    try:
        import torch
        if torch.cuda.is_available():
            print(f"\n✅ PyTorch {torch.__version__} — CUDA available ({torch.version.cuda})")
            total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
            if total_gb >= 14:
                print("   Recommendation: Use 1.7B models for best quality")
            else:
                print("   Recommendation: Use 0.6B models to stay within VRAM")
        else:
            print("\n⚠️  PyTorch installed but CUDA not available. Check your runtime type.")
    except ImportError:
        print("\nℹ️  PyTorch not yet installed (run Section 1 first).")

print("\n" + "=" * 60)

---
## Section 1 — 📦 Install Dependencies

| Package | Purpose |
|---|---|
| `qwen-tts` | Official Qwen3-TTS Python package |
| `soundfile` | Read/write WAV files |
| `gradio` | Web UI (Section 7) |
| `flash-attn` | *(Optional)* Reduces VRAM ~20%, faster inference |

> **💡 Flash Attention** is optional. Set `INSTALL_FLASH_ATTN = True` below if you want it.  
> It compiles from source and takes **5–10 minutes** the first time. Leave `False` to skip.

In [ ]:
# ─── Toggle Flash Attention install ───────────────────────────────────────────
INSTALL_FLASH_ATTN = False   # Set True to install (takes 5–10 min, compiles from source)
# ──────────────────────────────────────────────────────────────────────────────

print("📦 Installing core dependencies...")
!pip install -q -U qwen-tts soundfile gradio
print("✅ Core packages installed!")

if INSTALL_FLASH_ATTN:
    print("\n⚡ Installing Flash Attention 2 (this compiles from source — ~5–10 min)...")
    !MAX_JOBS=4 pip install -q -U flash-attn --no-build-isolation
    print("✅ Flash Attention installed!")
else:
    print("\nℹ️  Flash Attention skipped (INSTALL_FLASH_ATTN=False). Using SDPA instead.")
    print("   This is fine for free Colab — SDPA is PyTorch's built-in efficient attention.")

print("\n🎉 Ready! Proceed to Section 2 to configure settings.")

---
## Section 2 — ⚙️ Configuration
**Edit this cell** to configure the entire notebook. All other sections read from these variables.

| Setting | Options | Notes |
|---|---|---|
| `MODEL_SIZE` | `"1.7B"` / `"0.6B"` | 1.7B = higher quality; 0.6B = faster, less VRAM |
| `DTYPE` | `"bfloat16"` / `"float16"` | bfloat16 is preferred; use float16 if errors occur |
| `USE_FLASH` | `True` / `False` | Set True only if you installed flash-attn in Section 1 |

In [ ]:
import torch
import soundfile as sf
import gc
import os
from IPython.display import Audio, display

# ─── USER CONFIGURATION ────────────────────────────────────────────────────────
MODEL_SIZE  = "1.7B"       # "0.6B" for speed/low VRAM, "1.7B" for best quality
DTYPE       = "bfloat16"   # "bfloat16" (recommended) or "float16"
USE_FLASH   = False        # True only if you installed flash-attn in Section 1
# ──────────────────────────────────────────────────────────────────────────────

# Resolve dtype
DTYPE_MAP   = {"bfloat16": torch.bfloat16, "float16": torch.float16}
TORCH_DTYPE = DTYPE_MAP[DTYPE]

# Resolve attention implementation
ATTN_IMPL   = "flash_attention_2" if USE_FLASH else "sdpa"

# Output directory
OUTPUT_DIR  = "/content/qwen3_tts_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Utility: clear VRAM before loading a new model
def clear_vram(model_var=None):
    """Delete a model and free GPU memory."""
    if model_var is not None:
        del model_var
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
        print(f"   🧹 VRAM freed — {free/1e9:.1f} GB available")

# Utility: play and save audio
def play_and_save(wavs, sr, filename):
    """Save the first wav in the list and play it in the notebook."""
    path = os.path.join(OUTPUT_DIR, filename)
    sf.write(path, wavs[0], sr)
    print(f"   💾 Saved to: {path}")
    display(Audio(path, autoplay=True))
    return path

# Utility: chunk long text and concatenate audio
def generate_long_text(model_fn, text, max_chars=300, **kwargs):
    """
    Split long text into sentence chunks and concatenate audio.
    model_fn: a callable like model.generate_voice_design or model.generate_custom_voice
    """
    import re
    import numpy as np
    # Split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) + 1 <= max_chars:
            current = (current + " " + s).strip()
        else:
            if current:
                chunks.append(current)
            current = s
    if current:
        chunks.append(current)

    print(f"   📝 Splitting into {len(chunks)} chunk(s)...")
    all_audio, sr = [], None
    for i, chunk in enumerate(chunks):
        print(f"   [{i+1}/{len(chunks)}] Generating: {chunk[:60]}...")
        wavs, sr = model_fn(text=chunk, **kwargs)
        all_audio.append(wavs[0])

    combined = np.concatenate(all_audio)
    return [combined], sr

print("=" * 55)
print("  Configuration loaded!")
print("=" * 55)
print(f"  Model size   : {MODEL_SIZE}")
print(f"  Dtype        : {DTYPE}")
print(f"  Attention    : {ATTN_IMPL}")
print(f"  Output dir   : {OUTPUT_DIR}")
print("=" * 55)
print("\n✅ Ready — proceed to any section below!")

---
## Section 3 — 🎙️ Custom Voice (Preset Speakers)

Uses the **CustomVoice** model — choose from a set of built-in speaker personas.  
Great for quick, high-quality TTS without any reference audio.

**Model:** `Qwen3-TTS-12Hz-{0.6B|1.7B}-CustomVoice`

In [ ]:
# ─── Load CustomVoice model ────────────────────────────────────────────────────
from qwen_tts import Qwen3TTSModel

print(f"⏳ Loading Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice...")
print("   (First run downloads the model — may take a few minutes)")

cv_model = Qwen3TTSModel.from_pretrained(
    f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-CustomVoice",
    device_map="cuda:0",
    dtype=TORCH_DTYPE,
    attn_implementation=ATTN_IMPL,
)

print("\n✅ Model loaded!")
print("\n📋 Available speakers:")
speakers = cv_model.get_supported_speakers()
for i, sp in enumerate(speakers):
    print(f"   {i+1:2}. {sp}")

print("\n🌐 Supported languages:")
langs = cv_model.get_supported_languages()
print("   ", ", ".join(langs))

In [ ]:
# ─── Single inference ─────────────────────────────────────────────────────────
# Edit these to try different speakers, texts, and instructions

TEXT_CV      = "Welcome! This is Qwen3-TTS running on Google Colab. It's free, open-source, and incredibly powerful."
SPEAKER_CV   = "Ryan"           # Change to any speaker from the list above
LANGUAGE_CV  = "English"        # e.g. "English", "Chinese", "Japanese", "French"
INSTRUCT_CV  = "Speak warmly and with genuine enthusiasm, like a friendly product demo."  # Leave "" to omit

print(f"🎙️  Generating speech with speaker: {SPEAKER_CV}")
print(f"   Text     : {TEXT_CV[:80]}...")
print(f"   Instruct : {INSTRUCT_CV}")

wavs, sr = cv_model.generate_custom_voice(
    text=TEXT_CV,
    language=LANGUAGE_CV,
    speaker=SPEAKER_CV,
    instruct=INSTRUCT_CV,
)

print(f"\n✅ Generated! Sample rate: {sr} Hz")
play_and_save(wavs, sr, "custom_voice_single.wav")

In [ ]:
# ─── Batch inference — two speakers at once ────────────────────────────────────
print("🎙️  Batch generation with 2 different speakers...")

batch_texts    = [
    "The quick brown fox jumped over the lazy dog — a classic pangram!",
    "In the beginning, the universe was created. This has made a lot of people very angry.",
]
batch_speakers = ["Ryan", "Vivian"]   # One speaker per text
batch_langs    = ["English", "English"]
batch_instructs= ["Calm and clear narration.", "Slightly amused, as if recounting a funny story."]

wavs_batch, sr = cv_model.generate_custom_voice(
    text=batch_texts,
    language=batch_langs,
    speaker=batch_speakers,
    instruct=batch_instructs,
)

for i, (wav, spk) in enumerate(zip(wavs_batch, batch_speakers)):
    path = os.path.join(OUTPUT_DIR, f"custom_voice_batch_{i+1}_{spk}.wav")
    sf.write(path, wav, sr)
    print(f"\n   [{i+1}] Speaker: {spk} — {batch_texts[i][:60]}")
    print(f"       💾 Saved to: {path}")
    display(Audio(path, autoplay=False))

print("\n✅ Batch generation complete!")

---
## Section 4 — 🎨 Voice Design

Describe the voice you want in **plain English** (or any supported language).  
The model creates a completely new voice persona matching your description.

**Model:** `Qwen3-TTS-12Hz-1.7B-VoiceDesign` *(always 1.7B — no 0.6B variant)*

### 💡 Example Instruct Prompts
| Style | Instruct |
|---|---|
| Documentary narrator | `A deep, calm, measured male narrator — authoritative yet approachable` |
| Excited friend | `A bubbly, energetic young woman speaking at a slightly faster pace with enthusiasm` |
| Villain | `A low, silky, menacing voice with deliberate pacing and a slight British accent` |
| Whispered secret | `Whispering softly and conspiratorially, barely above breath` |
| Children's story | `A warm, gentle storyteller's voice — slow, clear, and wonderfully expressive` |

In [ ]:
# ─── Clear previous model from VRAM ───────────────────────────────────────────
try:
    print("🧹 Clearing previous model from VRAM...")
    clear_vram(cv_model)
    del cv_model
except NameError:
    pass  # No previous model to clear

# ─── Load VoiceDesign model ───────────────────────────────────────────────────
# NOTE: VoiceDesign is only available in 1.7B regardless of MODEL_SIZE setting
print("\n⏳ Loading Qwen3-TTS-12Hz-1.7B-VoiceDesign...")
print("   (First run downloads the model — may take a few minutes)")

vd_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",   # Only 1.7B has VoiceDesign
    device_map="cuda:0",
    dtype=TORCH_DTYPE,
    attn_implementation=ATTN_IMPL,
)

print("\n✅ VoiceDesign model loaded! Proceed to the next cell.")

In [ ]:
# ─── Voice Design — Single inference ──────────────────────────────────────────
# Edit TEXT_VD and INSTRUCT_VD to design your own voice!

TEXT_VD    = "The stars above us are ancient light, traveling billions of years just to reach your eyes tonight."
INSTRUCT_VD = "A deep, calm, documentary-narrator voice — measured pace, thoughtful gravitas, like David Attenborough."
LANGUAGE_VD = "English"

# ─── Try one of these presets by copy-pasting into INSTRUCT_VD above ──────────
# PRESET_1 = "A bubbly, energetic young woman speaking fast with contagious enthusiasm"
# PRESET_2 = "A low, silky, menacing voice with deliberate pacing and a faint British accent"
# PRESET_3 = "Whispering softly and conspiratorially, barely above breath"
# PRESET_4 = "A warm, gentle storyteller — slow, clear, and wonderfully expressive"
# PRESET_5 = "An old wise wizard, gravelly and slow, with the weight of centuries"

print(f"🎨 Designing voice with instruction:")
print(f"   {INSTRUCT_VD}")
print(f"\n   Text: {TEXT_VD[:80]}...")

wavs, sr = vd_model.generate_voice_design(
    text=TEXT_VD,
    language=LANGUAGE_VD,
    instruct=INSTRUCT_VD,
)

print(f"\n✅ Generated! Sample rate: {sr} Hz")
play_and_save(wavs, sr, "voice_design_output.wav")

In [ ]:
# ─── Voice Design — Batch: compare two different voice personas ────────────────
print("🎨 Batch: Generating two contrasting voice designs...\n")

SAME_TEXT = "Breaking news: scientists have discovered that the universe smells faintly of raspberries."

batch_vd_instructs = [
    "A serious, authoritative TV news anchor — clear diction, no emotion.",
    "A comedic radio host — can barely contain laughter, upbeat and playful.",
]

wavs_vd, sr = vd_model.generate_voice_design(
    text=[SAME_TEXT, SAME_TEXT],
    language=["English", "English"],
    instruct=batch_vd_instructs,
)

labels = ["news_anchor", "comedy_host"]
for i, (wav, label) in enumerate(zip(wavs_vd, labels)):
    path = os.path.join(OUTPUT_DIR, f"voice_design_{label}.wav")
    sf.write(path, wav, sr)
    print(f"  [{i+1}] {label}: {batch_vd_instructs[i]}")
    print(f"       💾 Saved: {path}")
    display(Audio(path, autoplay=False))
    print()

print("✅ Compare the two voices above — same text, completely different delivery!")

---
## Section 5 — 🧬 Voice Cloning

Clone **any voice** from a short (3–15 second) reference audio clip.  
Provide the reference audio + an accurate transcript — the model does the rest.

**Model:** `Qwen3-TTS-12Hz-{0.6B|1.7B}-Base`

### 📁 Reference Audio Options
- **Option A (default):** Uses the official Qwen demo WAV (English female voice)
- **Option B:** Upload your own WAV file from your computer
- **Option C:** Provide a URL to any public WAV/MP3

> **💡 Tips for good cloning quality:**
> - Use **3–15 seconds** of clean, noise-free speech
> - Provide an **accurate transcript** of what's said in the reference
> - Avoid music or background noise in the reference

In [ ]:
# ─── Clear previous model from VRAM ───────────────────────────────────────────
try:
    print("🧹 Clearing previous model from VRAM...")
    clear_vram(vd_model)
    del vd_model
except NameError:
    pass

# ─── Load Base model ──────────────────────────────────────────────────────────
print(f"\n⏳ Loading Qwen3-TTS-12Hz-{MODEL_SIZE}-Base...")
print("   (First run downloads the model — may take a few minutes)")

base_model = Qwen3TTSModel.from_pretrained(
    f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base",
    device_map="cuda:0",
    dtype=TORCH_DTYPE,
    attn_implementation=ATTN_IMPL,
)

print("\n✅ Base model loaded! Proceed to the next cell.")

In [ ]:
# ─── Choose your reference audio ──────────────────────────────────────────────
# OPTION A: Official Qwen demo audio (works out of the box — English female)
REF_AUDIO = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-TTS-Repo/clone.wav"
REF_TEXT  = "Okay. Yeah. I resent you. I love you. I respect you. But you know what? You blew it! And thanks to you."

# OPTION B: Upload your own file
# Uncomment the block below and comment out OPTION A above
# ─────────────────────────────────────────────────────────────────────────────
# from google.colab import files
# print("📂 Select a WAV file to upload as your voice reference:")
# uploaded = files.upload()
# REF_AUDIO = list(uploaded.keys())[0]
# REF_TEXT  = ""  # IMPORTANT: Type the EXACT transcript of what's said in your audio
# ─────────────────────────────────────────────────────────────────────────────

# OPTION C: Any public URL to a WAV/MP3
# REF_AUDIO = "https://your-url.com/sample.wav"
# REF_TEXT  = "The exact words spoken in that audio file."

# ─── Listen to the reference before cloning ───────────────────────────────────
print("🔊 Reference audio preview:")
print(f"   Source  : {str(REF_AUDIO)[:80]}")
print(f"   Transcript: {REF_TEXT}")

if isinstance(REF_AUDIO, str) and REF_AUDIO.startswith("http"):
    display(Audio(REF_AUDIO))
elif isinstance(REF_AUDIO, str) and os.path.exists(REF_AUDIO):
    display(Audio(REF_AUDIO))

In [ ]:
# ─── Generate cloned speech ────────────────────────────────────────────────────
TARGET_TEXT = "I have been cloned by Qwen3-TTS. This voice was captured from just a few seconds of reference audio, and now I can say whatever you type here."
TARGET_LANG = "English"

print("🧬 Cloning voice...")
print(f"   Target text: {TARGET_TEXT[:80]}...")

wavs, sr = base_model.generate_voice_clone(
    text=TARGET_TEXT,
    language=TARGET_LANG,
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
)

print(f"\n✅ Voice cloned! Sample rate: {sr} Hz")
play_and_save(wavs, sr, "voice_clone_output.wav")

In [ ]:
# ─── Reusable clone prompt — efficient batch generation ───────────────────────
# create_voice_clone_prompt builds the reference features ONCE, then reuses
# them across many sentences — much faster than re-extracting each time.

print("🧬 Building reusable voice clone prompt...")
voice_clone_prompt = base_model.create_voice_clone_prompt(
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    x_vector_only_mode=False,  # True = faster but lower quality
)
print("✅ Clone prompt created!\n")

# Now generate multiple sentences without re-extracting the reference
sentences = [
    "Chapter one: The journey begins on a cold morning in November.",
    "She had always known this day would come, though she never imagined it would arrive so soon.",
    "The sky was the color of old pewter, and the air smelled of rain and pine needles.",
]

print("🎬 Generating a 3-sentence story with the cloned voice:")
wavs_batch, sr = base_model.generate_voice_clone(
    text=sentences,
    language=["English"] * len(sentences),
    voice_clone_prompt=voice_clone_prompt,
)

import numpy as np
for i, (wav, sent) in enumerate(zip(wavs_batch, sentences)):
    path = os.path.join(OUTPUT_DIR, f"voice_clone_sentence_{i+1}.wav")
    sf.write(path, wav, sr)
    print(f"\n  [{i+1}] {sent}")
    display(Audio(path, autoplay=False))

# Also save a single concatenated file
combined_path = os.path.join(OUTPUT_DIR, "voice_clone_story_combined.wav")
sf.write(combined_path, np.concatenate(wavs_batch), sr)
print(f"\n📖 Full story (concatenated): {combined_path}")
display(Audio(combined_path, autoplay=False))
print("\n✅ Batch cloning complete!")

---
## Section 6 — 🔄 Voice Design → Clone (Advanced Workflow)

This advanced workflow lets you **invent a voice** with text instructions, then **clone it** for consistent multi-sentence generation — without ever recording a real person.

**Flow:**
```
  Describe voice in words
         │
         ▼
  VoiceDesign model → reference_clip.wav
         │
         ▼
  Base model (create_voice_clone_prompt) → reusable prompt
         │
         ▼
  Base model (generate_voice_clone) → many consistent sentences
```

> **Use case:** Audiobooks, games, podcasts — create a consistent fictional character voice.

In [ ]:
# ─── Clear previous model from VRAM ───────────────────────────────────────────
try:
    print("🧹 Clearing previous model from VRAM...")
    clear_vram(base_model)
    del base_model
except NameError:
    pass

print()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: Load VoiceDesign model and create a reference clip
# ═══════════════════════════════════════════════════════════════════════════════
print("STEP 1: Loading VoiceDesign model...")
vd_model2 = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=TORCH_DTYPE,
    attn_implementation=ATTN_IMPL,
)
print("✅ VoiceDesign model ready")

# Describe the character voice
CHARACTER_INSTRUCT = (
    "A wise, elderly scholar — unhurried, warm, with slight roughness in the voice "
    "hinting at decades of storytelling. British accent, thoughtful pauses."
)
REF_SENTENCE = (
    "You know, the most extraordinary thing about knowledge is not how much of it there is, "
    "but how little of it we ever truly need."
)

print(f"\n🎨 Designing character voice:")
print(f"   Instruct: {CHARACTER_INSTRUCT}")

ref_wavs, sr = vd_model2.generate_voice_design(
    text=REF_SENTENCE,
    language="English",
    instruct=CHARACTER_INSTRUCT,
)

ref_path = os.path.join(OUTPUT_DIR, "design_then_clone_reference.wav")
sf.write(ref_path, ref_wavs[0], sr)
print(f"\n✅ Reference clip created!")
print(f"   💾 {ref_path}")
display(Audio(ref_path, autoplay=True))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: Load Base model and build a reusable clone prompt
# ═══════════════════════════════════════════════════════════════════════════════
print("STEP 2: Swapping to Base model for cloning...")
clear_vram(vd_model2)
del vd_model2

base_model2 = Qwen3TTSModel.from_pretrained(
    f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-Base",
    device_map="cuda:0",
    dtype=TORCH_DTYPE,
    attn_implementation=ATTN_IMPL,
)
print("✅ Base model loaded")

print("\n🔗 Building reusable clone prompt from the designed voice...")
char_prompt = base_model2.create_voice_clone_prompt(
    ref_audio=(ref_wavs[0], sr),   # pass numpy array + sample rate directly
    ref_text=REF_SENTENCE,
)
print("✅ Character voice prompt ready!")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: Generate a multi-sentence audiobook excerpt with the character voice
# ═══════════════════════════════════════════════════════════════════════════════
print("\nSTEP 3: Generating consistent multi-sentence content...")

audiobook_lines = [
    "And so it was that on the first day of winter, the old professor opened the door to his library.",
    "The shelves rose to the ceiling, dense with books that smelled of dust and forgotten adventures.",
    "He smiled, settled into his chair, and began to read — as he had done every single morning for fifty years.",
    "Some men collect gold. Some collect power. He collected questions.",
]

wavs_char, sr = base_model2.generate_voice_clone(
    text=audiobook_lines,
    language=["English"] * len(audiobook_lines),
    voice_clone_prompt=char_prompt,
)

print("\n📖 Audiobook excerpt — consistent character voice:")
import numpy as np
all_audio = []
for i, (wav, line) in enumerate(zip(wavs_char, audiobook_lines)):
    path = os.path.join(OUTPUT_DIR, f"audiobook_line_{i+1}.wav")
    sf.write(path, wav, sr)
    all_audio.append(wav)
    print(f"\n  [{i+1}] {line}")
    display(Audio(path, autoplay=False))

# Save full combined audiobook excerpt
combined = np.concatenate(all_audio)
full_path = os.path.join(OUTPUT_DIR, "audiobook_excerpt_combined.wav")
sf.write(full_path, combined, sr)
print(f"\n🎧 Full excerpt (combined): {full_path}")
display(Audio(full_path, autoplay=False))
print("\n✅ Design → Clone workflow complete!")

---
## Section 7 — 🌐 Gradio Web UI (Optional)

Launch the **official `qwen-tts-demo`** — a full-featured interactive web interface.  
Generates a **public URL** you can share or use from any browser.

Choose which model to demo:
- `CustomVoice` — pick preset speakers
- `VoiceDesign` — type a voice description
- `Base` — upload reference audio and clone

> **⚠️ Note:** Running the Gradio UI loads a model into VRAM. Run the VRAM clear cell below first if coming from Sections 3–6.

In [ ]:
# ─── Clear VRAM before launching Gradio ───────────────────────────────────────
for var_name in ["cv_model", "vd_model", "vd_model2", "base_model", "base_model2"]:
    if var_name in dir():
        clear_vram(globals().get(var_name))
        globals().pop(var_name, None)

gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"✅ VRAM cleared — {free/1e9:.1f} GB available for Gradio")

In [ ]:
import subprocess, time

# ─── Choose which model to demo ───────────────────────────────────────────────
GRADIO_MODEL_TYPE = "CustomVoice"   # "CustomVoice", "VoiceDesign", or "Base"
GRADIO_MODEL_SIZE = "1.7B"          # "0.6B" or "1.7B"
GRADIO_PORT       = 7860
# ──────────────────────────────────────────────────────────────────────────────

GRADIO_MODEL_ID = f"Qwen/Qwen3-TTS-12Hz-{GRADIO_MODEL_SIZE}-{GRADIO_MODEL_TYPE}"

print(f"🌐 Launching Gradio UI for: {GRADIO_MODEL_ID}")
print(f"   Port: {GRADIO_PORT}")
print("   A public URL will appear below — open it in any browser.\n")

# Launch via the bundled CLI tool (comes with qwen-tts)
demo_proc = subprocess.Popen(
    [
        "qwen-tts-demo",
        GRADIO_MODEL_ID,
        "--ip", "0.0.0.0",
        "--port", str(GRADIO_PORT),
        "--share",   # Generates a public gradio.live URL
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream output until the public URL appears
for line in demo_proc.stdout:
    print(line, end="")
    if "gradio.live" in line or "Running on public URL" in line:
        print("\n✅ Gradio UI is live — click the URL above!")
        print("   (This cell will keep running while the demo is active.)")
        break

# Keep streaming output (interrupt kernel to stop)
for line in demo_proc.stdout:
    print(line, end="")

---
## Section 8 — 💾 Save & Download Outputs

All generated audio is saved in `/content/qwen3_tts_outputs/`.  
Use the cells below to **list**, **download**, or **save to Google Drive**.

In [ ]:
# ─── List all generated WAV files ─────────────────────────────────────────────
import glob

wav_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.wav")))

print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"   Found {len(wav_files)} WAV file(s):\n")

total_size = 0
for f in wav_files:
    size = os.path.getsize(f)
    total_size += size
    # Get duration
    data, rate = sf.read(f)
    duration = len(data) / rate
    print(f"   📄 {os.path.basename(f):45s}  {duration:5.1f}s  {size/1024:.0f} KB")

print(f"\n   Total: {total_size/1024:.0f} KB")

In [ ]:
# ─── Option A: Download files directly to your computer ───────────────────────
# This will download ALL generated WAV files as a zip archive.

import shutil
from google.colab import files

# Create a zip of all outputs
zip_path = "/content/qwen3_tts_outputs"
shutil.make_archive("/content/qwen3_tts_outputs", "zip", OUTPUT_DIR)

print("📦 Zipping outputs...")
zip_size = os.path.getsize("/content/qwen3_tts_outputs.zip") / 1024
print(f"✅ Archive created: qwen3_tts_outputs.zip ({zip_size:.0f} KB)")
print("\n⬇️  Starting download...")
files.download("/content/qwen3_tts_outputs.zip")

In [ ]:
# ─── Option B: Save to Google Drive ───────────────────────────────────────────
# Mounts your Drive and copies all WAV files to a dedicated folder.

from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_FOLDER = "/content/drive/MyDrive/Qwen3-TTS-Outputs"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

wav_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.wav")))
print(f"\n☁️  Copying {len(wav_files)} file(s) to Google Drive...")

for f in wav_files:
    dest = os.path.join(DRIVE_FOLDER, os.path.basename(f))
    shutil.copy2(f, dest)
    print(f"   ✓  {os.path.basename(f)}")

print(f"\n✅ All files saved to: {DRIVE_FOLDER}")
print("   Access them at: Google Drive → MyDrive → Qwen3-TTS-Outputs")

---
## 💡 Tips & Troubleshooting

### VRAM / OOM Errors
- Switch to **0.6B models** in Section 2: `MODEL_SIZE = "0.6B"`
- Make sure to run the VRAM clear cells between sections
- If still OOM: **Runtime → Restart runtime**, then rerun from Section 2

### Slow Generation on Long Text
- Use the `generate_long_text()` helper defined in Section 2
- This splits your text into sentence-sized chunks and concatenates the audio

### Voice Cloning Quality
- Reference audio should be **3–15 seconds** of clean speech
- Provide an **exact, accurate transcript** for best results
- Try `x_vector_only_mode=True` in `create_voice_clone_prompt` for faster (but slightly lower quality) cloning

### Flash Attention
- If you want to install it: set `INSTALL_FLASH_ATTN = True` in Section 1, run it, then set `USE_FLASH = True` in Section 2
- Saves ~20% VRAM and speeds up generation

### Session Disconnection
- Free Colab disconnects after ~90 minutes of idle time
- Save outputs to Drive (Section 8) regularly
- Model weights are cached in `/root/.cache/huggingface` — reconnecting requires a quick re-download

### Official Resources
- 📦 [GitHub Repo](https://github.com/QwenLM/Qwen3-TTS)
- 🤗 [HuggingFace Collection](https://huggingface.co/collections/Qwen/qwen3-tts)
- 🖥️ [Live HF Demo](https://huggingface.co/spaces/Qwen/Qwen3-TTS-Demo)
- 📑 [Technical Paper](https://arxiv.org/abs/2601.15621)